# Лабораторна №0: RAG-система про Чернівці

Мета: побудувати систему, яка відповідає на питання про Чернівці та регіон, використовуючи статті Wikipedia як джерело знань.

Стек: LangChain + OpenAI (`gpt-4o-mini`, `text-embedding-3-small`) + FAISS.

## Секція 0. Налаштування

Завантажуємо ключі з `.env`, перевіряємо наявність `OPENAI_API_KEY` та кількість документів у `data/`.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY не знайдено в .env"
print(f"OpenAI key: ...{OPENAI_API_KEY[-4:]} (перевірка)")

DATA_DIR = Path("../data")
print(f"Документів у data/: {len(list(DATA_DIR.glob('*.txt')))}")

OpenAI key: ...u-EA (перевірка)
Документів у data/: 4


## Секція 1. Завантаження документів

Читаємо всі `.txt` файли з папки `data/` через `DirectoryLoader`.

In [2]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader

loader = DirectoryLoader(
    str(DATA_DIR),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = loader.load()

print(f"Завантажено документів: {len(documents)}")
for doc in documents:
    print(f"  - {Path(doc.metadata['source']).name}: {len(doc.page_content)} симв.")

Завантажено документів: 4
  - kobylyanska.txt: 31984 симв.
  - chernivtsi_city.txt: 91678 симв.
  - chernivtsi_university.txt: 17660 симв.
  - bukovyna.txt: 63401 симв.


## Секція 2. Chunking — експерименти з трьома стратегіями

Порівнюємо:

1. `CharacterTextSplitter` — просте побиття по символу `\n`.
2. `RecursiveCharacterTextSplitter` з розміром 1000/100 — зберігає абзаци та речення.
3. `RecursiveCharacterTextSplitter` з розміром 500/50 — дрібніші блоки для точнішого retrieval.

In [3]:
from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

splitter_1 = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100, separator="\n")
chunks_1 = splitter_1.split_documents(documents)

splitter_2 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks_2 = splitter_2.split_documents(documents)

splitter_3 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks_3 = splitter_3.split_documents(documents)

for i, chunks in enumerate([chunks_1, chunks_2, chunks_3], 1):
    avg = sum(len(c.page_content) for c in chunks) / len(chunks)
    print(f"Стратегія {i}: {len(chunks)} chunks, середня довжина {avg:.0f} симв.")

Created a chunk of size 1160, which is longer than the specified 1000
Created a chunk of size 1957, which is longer than the specified 1000
Created a chunk of size 1235, which is longer than the specified 1000
Created a chunk of size 1123, which is longer than the specified 1000
Created a chunk of size 1125, which is longer than the specified 1000
Created a chunk of size 1022, which is longer than the specified 1000
Created a chunk of size 1515, which is longer than the specified 1000
Created a chunk of size 1315, which is longer than the specified 1000
Created a chunk of size 1064, which is longer than the specified 1000
Created a chunk of size 1604, which is longer than the specified 1000
Created a chunk of size 1003, which is longer than the specified 1000
Created a chunk of size 1023, which is longer than the specified 1000
Created a chunk of size 1036, which is longer than the specified 1000
Created a chunk of size 1088, which is longer than the specified 1000
Created a chunk of s

Стратегія 1: 243 chunks, середня довжина 855 симв.
Стратегія 2: 305 chunks, середня довжина 685 симв.
Стратегія 3: 620 chunks, середня довжина 341 симв.


Для основного індексу обираємо **стратегію 2** (Recursive, 1000/100) — вона зберігає межі абзаців і дає збалансовану кількість chunks.

## Секція 3. Embeddings та FAISS-індекс

Перетворюємо chunks у вектори через `text-embedding-3-small` та будуємо FAISS-індекс для швидкого пошуку.

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chunks = chunks_2

vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"FAISS index: {vectorstore.index.ntotal} векторів")

FAISS index: 305 векторів


## Секція 4. RAG-ланцюжок

`RetrievalQA` поєднує retriever (FAISS) та LLM (`gpt-4o-mini`). Для кожного питання retriever віддає `k=4` найрелевантніші chunks, які потім підставляються у prompt для LLM.

In [5]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
)

print("RAG chain готовий.")

RAG chain готовий.


## Секція 5. Тестування — 5 питань

Задаємо системі питання з різних областей (місто, університет, особистості, регіон). Для кожної відповіді виводимо, з яких джерел LLM узяла інформацію.

In [6]:
questions = [
    "Коли заснували Чернівецький національний університет?",
    "Хто така Ольга Кобилянська та чим вона відома?",
    "Які райони входять до історичної Буковини?",
    "Яка архітектурна пам'ятка Чернівців увійшла до списку UNESCO?",
    "Скільки населення у Чернівцях?",
]

for q in questions:
    print(f"\nQ: {q}")
    result = qa_chain.invoke({"query": q})
    print(f"A: {result['result']}")
    sources = [Path(d.metadata['source']).name for d in result['source_documents']]
    print(f"   Джерела: {sources}")


Q: Коли заснували Чернівецький національний університет?
A: Чернівецький національний університет імені Юрія Федьковича був заснований 4 жовтня 1875 року.
   Джерела: ['bukovyna.txt', 'chernivtsi_city.txt', 'chernivtsi_university.txt', 'chernivtsi_university.txt']

Q: Хто така Ольга Кобилянська та чим вона відома?
A: Ольга Кобилянська — українська письменниця-модерністка, яка народилася 27 листопада 1863 року в Герцогстві Буковина (тепер частина Румунії) і померла 21 березня 1942 року в Чернівцях. Вона була ранньою буковинською феміністкою та близькою подругою Лесі Українки. Кобилянська відома своїми повістями та оповіданнями, в яких порушує проблеми українського жіноцтва та життя буковинського села на межі століть. 

Вона також була однією з найважливіших постатей раннього модернізму в українській літературі. Незважаючи на обмежену офіційну освіту (лише 4 класи), Кобилянська активно боролася за права жінок і була однією з засновниць «Товариства руських жінок на Буковині» у 1894 році.

## Секція 6. Порівняння chunking-стратегій

Будуємо окремий RAG для кожної з трьох стратегій і запускаємо одне й те саме питання. Дивимось, чи суттєво відрізняються відповіді.

In [7]:
from langchain.chains import RetrievalQA


def build_qa(chunks_list):
    vs = FAISS.from_documents(chunks_list, embeddings)
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vs.as_retriever(search_kwargs={"k": 4}),
        return_source_documents=True,
    )


qa_by_strategy = {
    "1: CharacterTextSplitter, 1000/100": build_qa(chunks_1),
    "2: Recursive, 1000/100": build_qa(chunks_2),
    "3: Recursive, 500/50": build_qa(chunks_3),
}

test_questions = [
    "Опиши історію Чернівецького університету від заснування до сьогодні.",
    "Які твори написала Ольга Кобилянська та про що вони?",
    "Які історичні та природні райони входили до Буковини?",
]

for test_q in test_questions:
    print("\n" + "=" * 70)
    print(f"Q: {test_q}")
    print("=" * 70)
    for name, qa in qa_by_strategy.items():
        result = qa.invoke({"query": test_q})
        print(f"\n[{name}]")
        print(f"Довжина відповіді: {len(result['result'])} симв.")
        print(f"Retrieved chunks: {len(result['source_documents'])}")
        first_chunk = result["source_documents"][0].page_content[:120].replace("\n", " ")
        print(f"Перший chunk починається: «{first_chunk}...»")
        print(f"Відповідь:\n{result['result']}")



Q: Опиши історію Чернівецького університету від заснування до сьогодні.

[1: CharacterTextSplitter, 1000/100]
Довжина відповіді: 1757 симв.
Retrieved chunks: 4
Перший chunk починається: «Імператор Франц Йосиф I доволі прихильно ставився до Буковини. За активного сприяння цісаря було відкрито цілу низку нав...»
Відповідь:
Чернівецький університет, заснований 4 жовтня 1875 року за указом австрійського цісаря Франца Йосифа I, має багатий і цікавий шлях розвитку. Спочатку університет складався з трьох факультетів: філософського, теологічного та юридичного, і був відкритий на базі теологічного інституту, що існував з 1827 року. Першим ректором став Костянтин Томащук, український учений і громадський діяч.

Протягом свого існування університет став важливим освітнім і науковим центром не лише для Буковини, але й для інших регіонів, зокрема Галичини. У 1914 році в університеті навчалося близько 1,2 тисячі студентів, серед яких були українці, румуни, євреї та німці.

Під час Першої світової в

## Секція 7. Висновки

- **Що вдалося:** побудована повноцінна RAG-система на 4 Wikipedia-статтях. LLM відповідає на питання з посиланнями на джерела. Без RAG звичайний `gpt-4o-mini` міг би галюцинувати конкретні факти (наприклад, точний рік заснування університету).

- **Різниця між стратегіями:** `CharacterTextSplitter` розбиває механічно, часом ріже речення посередині. `RecursiveCharacterTextSplitter` краще зберігає семантичні межі. Менші chunks (500 симв.) дають точніший retrieval, але можуть втрачати контекст; більші (1000 симв.) — навпаки.

- **Обмеження:** якість відповідей залежить від якості джерел. Якщо у Wikipedia-статті немає інформації — LLM або чесно скаже «не знаю», або додасть щось зі свого pretraining'у (краще контролювати через prompt).

- **Наступні кроки (additional points):** загорнути систему в FastAPI-сервер, додати простий HTML-фронтенд з полем для питання та виводом відповіді + джерел.